# Experiment 9: CatBoost, and whether the blending plan is viable

Tuning is closed. LightGBM sits at CV 0.963275 and LB 0.964780, and the leaders are at
0.97102. The public forensics say the generator produced a smooth calibrated field with
no leak and no duplicate rows, and the top public notebooks are large stacks. That
points the remaining 0.0063 at ensembling rather than feature discovery.

CatBoost is the first diverse model. It handles the three categoricals with ordered
target statistics rather than LightGBM's split-based approach, which is a genuinely
different mechanism and the usual reason the two blend well.

## The number that decides the plan is not CatBoost's CV

CatBoost alone will probably land near LightGBM, possibly slightly under. That is not
the point and should not be read as failure. What matters is whether the two models
**disagree about which rows are hard**, because that is the only thing a blend can
exploit.

The metric is AUC, which reads only ordering, so the relevant measure is the **Spearman
rank correlation** between the two out-of-fold prediction vectors. Thresholds, fixed
here before the number exists:

- **below 0.97**: genuine disagreement. A rank blend should beat both. The plan is
  alive and the next models are worth building.
- **0.97 to 0.99**: marginal. A blend might buy a little. Proceed, but without
  expecting much.
- **above 0.99**: the two are the same model wearing different hats. Blending will buy
  nothing, and grinding out more GBDT variants would waste weeks. Say so and rethink.

The blend itself is also evaluated here, on the same folds, so the claim is tested
rather than assumed.

## Reproducibility

CatBoost gets the same treatment LightGBM needed, and none of it is assumed to carry
over. The configuration is trained twice and must come back bit-identical on both the
CV score and every out-of-fold prediction, before any of the real work runs.

In [ ]:
import csv
import time
from datetime import datetime, timezone
from pathlib import Path

import catboost as cb
import lightgbm as lgb
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold

SEED = 42
N_SPLITS = 5
TARGET = "addicted_label"
ID = "id"
CAT_COLS = ["gender", "stress_level", "academic_work_impact"]

# Matches the LightGBM working baseline: lr 0.05, budget lr*n = 100.
CB_KW = dict(iterations=2000, learning_rate=0.05, random_seed=SEED,
             thread_count=4, verbose=0, allow_writing_files=False)

print("catboost", cb.__version__, "| lightgbm", lgb.__version__)

In [ ]:
def locate():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "data" / "raw" / "train.csv").exists():
            return base, base / "data" / "raw"
    kag = Path("/kaggle/input/playground-series-s6e8")
    if (kag / "train.csv").exists():
        return Path("/kaggle/working"), kag
    raise FileNotFoundError("could not find train.csv")


REPO, RAW = locate()
SUB_DIR = REPO / "submissions"
OOF_DIR = REPO / "artifacts" / "oof"
SUB_DIR.mkdir(parents=True, exist_ok=True)
OOF_DIR.mkdir(parents=True, exist_ok=True)

train = pd.read_csv(RAW / "train.csv")
test = pd.read_csv(RAW / "test.csv")
sample = pd.read_csv(RAW / "sample_submission.csv")
FEATURES = [c for c in train.columns if c not in (ID, TARGET)]
y = train[TARGET].to_numpy()

# CatBoost needs categoricals as strings with no NaN, so missing becomes its own level.
# That is not an imputation choice: it keeps missingness as information rather than
# guessing a value, which matches what LightGBM's native routing already does.
Xtr, Xte = train[FEATURES].copy(), test[FEATURES].copy()
for c in CAT_COLS:
    Xtr[c] = Xtr[c].astype("object").fillna("__NA__").astype(str)
    Xte[c] = Xte[c].astype("object").fillna("__NA__").astype(str)
CAT_IDX = [FEATURES.index(c) for c in CAT_COLS]

assert ID not in FEATURES and TARGET not in FEATURES
assert not (set(train[ID]) & set(test[ID]))
assert list(FEATURES) == [c for c in test.columns if c != ID]

skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
folds = np.full(len(train), -1, dtype=int)
for i, (_, va) in enumerate(skf.split(train, y)):
    folds[va] = i
assert (folds >= 0).all()
print(f"{len(train):,} train rows, {len(FEATURES)} features, cat idx {CAT_IDX}")

In [ ]:
def run_catboost():
    oof = np.zeros(len(train), dtype=float)
    test_pred = np.zeros(len(test), dtype=float)
    fold_scores = []
    t0 = time.time()
    for f in range(N_SPLITS):
        tr_m, va_m = folds != f, folds == f
        model = cb.CatBoostClassifier(**CB_KW)
        model.fit(Xtr.loc[tr_m], y[tr_m], cat_features=CAT_IDX)
        p_va = model.predict_proba(Xtr.loc[va_m])[:, 1]
        oof[va_m] = p_va
        test_pred += model.predict_proba(Xte)[:, 1] / N_SPLITS
        fold_scores.append(roc_auc_score(y[va_m], p_va))
    return {"cv_mean": float(np.mean(fold_scores)), "cv_std": float(np.std(fold_scores)),
            "pooled": float(roc_auc_score(y, oof)), "secs": time.time() - t0,
            "oof": oof, "test_pred": test_pred}

## Determinism check, before anything else

In [ ]:
a = run_catboost()
b = run_catboost()
delta = abs(a["cv_mean"] - b["cv_mean"])
oof_max = float(np.abs(a["oof"] - b["oof"]).max())
print(f"run A CV : {a['cv_mean']:.9f}   ({a['secs']:.0f}s)")
print(f"run B CV : {b['cv_mean']:.9f}")
print(f"delta    : {delta:.12f}")
print(f"largest per-row OOF difference: {oof_max:.3e}")
assert delta == 0.0, f"CatBoost not deterministic: delta {delta:.3e}"
assert oof_max == 0.0, f"CatBoost OOF differs by up to {oof_max:.3e}"
print("\nbit-identical")
cat = a

## LightGBM at the same baseline, for a like-for-like comparison

Experiment 7's numbers are reused only if its saved OOF file is present. Otherwise it
is retrained here, because comparing against a differently-configured run is the
mistake that cost 21 minutes earlier.

In [ ]:
LGB_KW = dict(random_state=SEED, verbose=-1, deterministic=True,
              force_row_wise=True, n_jobs=4)
lgb_path = OOF_DIR / "lgbm_lr005_n2000_seed42.npy"

if lgb_path.exists():
    lgb_oof = np.load(lgb_path)
    print(f"loaded LightGBM OOF from {lgb_path.name}")
else:
    lgb_oof = np.zeros(len(train), dtype=float)
    for f in range(N_SPLITS):
        tr_m, va_m = folds != f, folds == f
        m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, **LGB_KW)
        m.fit(train.loc[tr_m, FEATURES], y[tr_m])
        lgb_oof[va_m] = m.predict_proba(train.loc[va_m, FEATURES])[:, 1]
    print("retrained LightGBM baseline")

lgb_cv = roc_auc_score(y, lgb_oof)
print(f"LightGBM pooled OOF AUC: {lgb_cv:.6f}")
print(f"CatBoost pooled OOF AUC: {cat['pooled']:.6f}")

## The decision number

In [ ]:
rank_corr = float(spearmanr(lgb_oof, cat["oof"]).statistic)
pear = float(np.corrcoef(lgb_oof, cat["oof"])[0, 1])
print(f"Spearman rank correlation of OOF predictions: {rank_corr:.6f}")
print(f"Pearson correlation (secondary):              {pear:.6f}\n")

if rank_corr < 0.97:
    print("VERDICT: genuine disagreement. A rank blend should beat both models.")
    print("The ensembling plan is alive. Build the next model family.")
elif rank_corr < 0.99:
    print("VERDICT: marginal diversity. A blend may buy a little. Proceed without")
    print("expecting much, and prefer a genuinely different model family next.")
else:
    print("VERDICT: these are the same model wearing different hats. Blending will")
    print("buy nothing. Do not grind out more GBDT variants. Rethink the approach.")

## Test the blend rather than assuming it

Rank average, not probability average. AUC reads only ordering, and rank averaging is
unaffected by the two models sitting on different probability scales.

In [ ]:
def to_rank(v):
    return pd.Series(v).rank(pct=True).to_numpy()


blend_oof = 0.5 * to_rank(lgb_oof) + 0.5 * to_rank(cat["oof"])
blend_cv = roc_auc_score(y, blend_oof)
best_single = max(lgb_cv, cat["pooled"])
print(f"LightGBM alone : {lgb_cv:.6f}")
print(f"CatBoost alone : {cat['pooled']:.6f}")
print(f"50/50 rank blend: {blend_cv:.6f}")
print(f"\ngain over the better single model: {blend_cv - best_single:+.6f}")
print(f"fold spread for reference: {cat['cv_std']:.6f}")
if blend_cv - best_single < cat["cv_std"]:
    print("\nThat gain is inside one fold standard deviation. Not established.")

In [ ]:
LEDGER = REPO / "experiments.csv"
COLUMNS = ["id", "utc", "name", "cv_mean", "cv_std", "folds",
           "lb_public", "lb_private", "submitted", "notes"]
rows = []
if LEDGER.exists():
    with LEDGER.open(newline="", encoding="utf-8") as fh:
        rows = list(csv.DictReader(fh))
next_id = max((int(r["id"]) for r in rows), default=0) + 1

np.save(OOF_DIR / f"catboost_lr005_n2000_seed{SEED}.npy", cat["oof"])
sub = sample.copy()
sub[TARGET] = cat["test_pred"]
sub.to_csv(SUB_DIR / f"catboost_lr005_n2000_seed{SEED}.csv", index=False)

rows.append({
    "id": str(next_id), "utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "name": "catboost_lr005", "cv_mean": f"{cat['cv_mean']:.6f}",
    "cv_std": f"{cat['cv_std']:.6f}", "folds": str(N_SPLITS),
    "lb_public": "", "lb_private": "", "submitted": "no",
    "notes": (f"catboost iterations=2000 lr=0.05, ordered target stats on cats, "
              f"spearman vs lgbm OOF {rank_corr:.4f}, 50/50 rank blend CV "
              f"{blend_cv:.6f}"),
})
with LEDGER.open("w", newline="", encoding="utf-8") as fh:
    w = csv.DictWriter(fh, fieldnames=COLUMNS)
    w.writeheader()
    w.writerows({c: r.get(c, "") for c in COLUMNS} for r in rows)
pd.read_csv(LEDGER)